# 00 — Ingestion des données : téléchargement et premier chargement de FD001

**Objectif de ce notebook** : récupérer le dataset NASA C-MAPSS depuis Kaggle et charger le sous-ensemble FD001 (`train`, `test`, `RUL`) sans aucune transformation, afin de vérifier que les fichiers sont complets et correctement structurés avant de passer à l'exploration (`01_exploration`).

Ce notebook ne fait aucun nettoyage ni calcul de cible : il se contente de charger et de constater. La construction de la RUL et le filtrage des capteurs se feront dans `02_preparation`.

**Source** : [behrad3d/nasa-cmaps sur Kaggle](https://www.kaggle.com/datasets/behrad3d/nasa-cmaps)

In [1]:
# --- Imports et configuration des chemins ---
import os
import zipfile
from pathlib import Path

import pandas as pd

# Racine du projet = deux niveaux au-dessus de ce notebook (notebooks/00_ingestion/ -> racine)
PROJECT_ROOT = Path.cwd().resolve().parents[1]
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET = "behrad3d/nasa-cmaps"
SUBSET = "FD001"  # on démarre par le sous-ensemble le plus simple

print(f"Racine du projet : {PROJECT_ROOT}")
print(f"Dossier de données brutes : {DATA_RAW}")

Racine du projet : C:\cmapss-prediction-rul
Dossier de données brutes : C:\cmapss-prediction-rul\data\raw


## 1. Récupération du dataset

Les données sont récupérées via l'API Kaggle et mises en cache localement dans `data/raw/` (téléchargement idempotent : ignoré si les fichiers sont déjà présents). Les prérequis d'authentification sont documentés dans le `README.md` du projet.

In [2]:
# --- Téléchargement du dataset (idempotent) ---
already_downloaded = any(DATA_RAW.rglob(f"*{SUBSET}*"))
kaggle_json = Path.home() / ".kaggle" / "kaggle.json"

if already_downloaded:
    print(f"Données {SUBSET} déjà présentes dans {DATA_RAW}, téléchargement ignoré.")
elif not kaggle_json.exists():
    raise EnvironmentError(
        "Authentification Kaggle non configurée. Voir la section Installation du README."
    )
else:
    # L'import est fait ici (et non en tête de notebook) : le module kaggle lit les
    # credentials dès l'import et lèverait une erreur si ce bloc n'était pas atteint en premier.
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files(KAGGLE_DATASET, path=str(DATA_RAW), unzip=True)

    print(f"Téléchargement terminé. Contenu de {DATA_RAW} :")
    for f in sorted(DATA_RAW.rglob("*")):
        if f.is_file():
            print(" -", f.relative_to(DATA_RAW))

Données FD001 déjà présentes dans C:\cmapss-prediction-rul\data\raw, téléchargement ignoré.


## 2. Chargement de FD001

Les fichiers `.txt` n'ont pas d'en-tête et sont séparés par des espaces multiples. On nomme explicitement les 26 colonnes selon la convention standard du dataset :
- `unit_number` : identifiant du moteur
- `time_in_cycles` : numéro du cycle
- `op_setting_1..3` : réglages opérationnels
- `sensor_1..21` : mesures capteurs

In [3]:
# --- Noms de colonnes standard du dataset C-MAPSS ---
COLUMN_NAMES = (
    ["unit_number", "time_in_cycles"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)


def find_file(pattern: str) -> Path:
    """Cherche un fichier par motif dans data/raw, quel que soit le sous-dossier créé par l'extraction du zip."""
    matches = list(DATA_RAW.rglob(pattern))
    if not matches:
        raise FileNotFoundError(
            f"Fichier '{pattern}' introuvable dans {DATA_RAW}. "
            "Vérifie que le téléchargement Kaggle (cellule précédente) s'est bien déroulé."
        )
    return matches[0]


# Les fichiers train/test n'ont pas d'en-tête et sont séparés par des espaces multiples.
df_train = pd.read_csv(find_file(f"train_{SUBSET}.txt"), sep=r"\s+", header=None, names=COLUMN_NAMES)
df_test = pd.read_csv(find_file(f"test_{SUBSET}.txt"), sep=r"\s+", header=None, names=COLUMN_NAMES)
df_rul = pd.read_csv(find_file(f"RUL_{SUBSET}.txt"), sep=r"\s+", header=None, names=["RUL"])

print(f"train_{SUBSET} : {df_train.shape}")
print(f"test_{SUBSET}  : {df_test.shape}")
print(f"RUL_{SUBSET}   : {df_rul.shape}")

train_FD001 : (20631, 26)
test_FD001  : (13096, 26)
RUL_FD001   : (100, 1)


## 3. Premier constat

On vérifie ici uniquement l'intégrité du chargement : aperçu, types, valeurs manquantes, cohérence du nombre de moteurs. L'analyse approfondie (distributions, tendances des capteurs) se fera dans `01_exploration`.

In [4]:
df_train.head()

,unit_number,time_in_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [5]:
# --- Types et valeurs manquantes ---
print("Types de colonnes :")
print(df_train.dtypes.value_counts())

n_missing_train = df_train.isna().sum().sum()
n_missing_test = df_test.isna().sum().sum()
print()
print(f"Valeurs manquantes — train : {n_missing_train}, test : {n_missing_test}")

Types de colonnes :
float64    22
int64       4
Name: count, dtype: int64

Valeurs manquantes — train : 0, test : 0


In [6]:
# --- Cohérence du nombre de moteurs ---
n_engines_train = df_train["unit_number"].nunique()
n_engines_test = df_test["unit_number"].nunique()
n_rul_rows = len(df_rul)

print(f"Moteurs distincts — train : {n_engines_train}, test : {n_engines_test}")
print(f"Lignes dans RUL_{SUBSET}.txt : {n_rul_rows}")

# Un RUL doit correspondre à exactement un moteur test
assert n_engines_test == n_rul_rows, "Incohérence : le nombre de moteurs test ne correspond pas au nombre de valeurs RUL."
print()
print("✓ Cohérence vérifiée : un RUL par moteur test.")

Moteurs distincts — train : 100, test : 100
Lignes dans RUL_FD001.txt : 100

✓ Cohérence vérifiée : un RUL par moteur test.


In [7]:
# --- Aperçu rapide des durées de vie (train) : nombre de cycles avant panne, par moteur ---
cycles_par_moteur = df_train.groupby("unit_number")["time_in_cycles"].max()

print("Durée de vie des moteurs (train), en cycles :")
print(cycles_par_moteur.describe())

Durée de vie des moteurs (train), en cycles :
count    100.000000
mean     206.310000
std       46.342749
min      128.000000
25%      177.000000
50%      199.000000
75%      229.250000
max      362.000000
Name: time_in_cycles, dtype: float64
